# Mule Pattern Learner — End-to-End Pipeline Walkthrough


## 0 · Setup & connect

In [1]:
# Make the src-layout package importable when running from the repo root.
import sys, pathlib
ROOT = pathlib.Path.cwd()
if (ROOT / "src").is_dir() and str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from mule_pattern_learner.tigergraph.client import Client
from mule_pattern_learner.tigergraph.settings import Settings

# Settings reads host / graphname / secret from .env (see tigergraph/settings.py).
client = Client(Settings())
print("connected:", client.graphname)

connected: Mule_Pattern_Learner


In [2]:
RUN_DESTRUCTIVE = False

# A tiny helper to keep long gsql logs readable in the notebook.
def show(text: str, n: int = 100) -> None:
    text = text.strip()
    print(text[:n] + ("\n... [truncated]" if len(text) > n else ""))

## 1 · The GSQL installer

In [4]:
from mule_pattern_learner.tigergraph import gsql_install
from mule_pattern_learner.tigergraph.gsql_paths import query_names, gsql_path

# Show the registry-key -> installed-name map so the mismatches are explicit.
print(f"{'registry key':28s} installed name")
for key in query_names():
    try:
        print(f"{key:28s} {gsql_install.get_query_from_file(key)}")
    except gsql_install.GsqlInstallError:
        print(f"{key:28s} (not a query — e.g. loading job)")

registry key                 installed name
account_account_degree       account_aa_degree_feature
cluster_with_wcc             tg_wcc_account_with_weights
derive_max_bins              derive_max_bins
derive_reference_epoch       derive_reference_epoch
diagnose_export              diagnose_export
diagnose_sentinels           diagnose_sentinels
export_account_features      export_account_features
export_edges_by_type         export_edges_by_type
export_has_paid_edges        export_has_paid_edges
fastrp                       tg_fastRP
fetch_account_features       fetch_account_features
fetch_has_paid_features      fetch_has_paid_features
get_masking_inputs           get_masking_inputs
get_split_accounts           get_split_accounts
identity_sharing             account_identity_sharing_features
loading_job                  (not a query — e.g. loading job)
match_parties                match_parties
money_flow                   account_money_flow_features
pagerank                     tg_pag

## 2 · Schema & load — INSPECT only

Your graph already has the schema and the aggregated edges/vertices loaded
(the raw ledger stays on disk; only `ACCOUNT_FLOW_AGG.csv` was loaded). We just
confirm the schema rather than recreate it.

In [6]:
# Read-only: list vertex & edge types and a few counts. No mutation.
vtypes = client.conn.getVertexTypes()
etypes = client.conn.getEdgeTypes()
print("vertex types:", vtypes)
print("edge types:", etypes)
print()
for vt in ("Account", "Party", "Resolved_Entity"):
    if vt in vtypes:
        cnt = client.conn.getVertexCount(vt, realtime=True)
        print(f"  {vt:18s} {cnt}")

vertex types: ['Account', 'Party', 'Device', 'IP', 'Phone', 'Email', 'Name', 'Birthdate', 'Street_Address', 'City', 'State', 'Postcode', 'Name_Minhash', 'Address_Minhash', 'Street_Minhash', 'Connected_Component', 'Resolved_Entity']
edge types: ['HAS_PAID', 'Account_Account', 'Party_Has_Account', 'Has_Device', 'Has_IP', 'Has_Phone', 'Has_Email', 'Has_Name', 'Has_Birthdate', 'Has_Street_Address', 'Has_City', 'Has_State', 'Has_Postcode', 'Has_Name_Minhash', 'Has_Address_Minhash', 'Has_Street_Minhash', 'Same_As', 'Party_In_Entity', 'Account_In_Ring']

  Account            191283
  Party              70000
  Resolved_Entity    68447


In [7]:
# The schema / loading_job source, for reference. Re-running these is the only
# truly destructive path; guarded so you can't run it by reflex.
if RUN_DESTRUCTIVE:
    raise SystemExit("Refusing: schema create + load would rebuild the graph. "
                     "Re-loading is an ~8h regeneration; do it deliberately, not here.")
else:
    print("schema.gsql / loading_job.gsql exist at:")
    print(" ", gsql_path("loading_job"))
    print("(inspect-only — graph already built)")

schema.gsql / loading_job.gsql exist at:
  /Users/abrahamchandy/Documents/Work/companies/2026/TigerGraph/MulePatternLearner/gsql/schema/loading_job.gsql
(inspect-only — graph already built)


## 3 · Community detection (WCC)

Two stages, in order: weight the Account–Account edges, then run weighted WCC,
which materializes `Connected_Component` vertices + `Account_In_Ring` edges and
writes `com_id` / `com_size` onto Accounts. **Mutates** community attributes —
gated, but safe to re-run (idempotent overwrite).

In [8]:
# Install both community queries from their .gsql files.
for key in ("weight_account_edges", "cluster_with_wcc"):
    log = gsql_install.install_query(client, key)
    print(f"installed {key} ->", gsql_install.installed_query_name(key))
    show(log, 200)
    print("-" * 40)

ReadTimeout: HTTPSConnectionPool(host='tg-59626898-34a0-406c-a5e5-a63b57d0dda4.tg-2635877100.i.tgcloud.io', port=443): Read timed out. (read timeout=None)

In [ ]:
# Run them in order (WCC depends on the weighted edges). Re-running overwrites
# com_id/com_size and re-materializes the components — idempotent, so allowed
# without the destructive gate, but flip to taste.
if RUN_DESTRUCTIVE:
    out1 = gsql_install.run_query(client, "weight_account_edges")
    print("weight_account_edges ->", out1)
    out2 = gsql_install.run_query(client, "cluster_with_wcc", {"min_link_weight": 2})
    print("cluster_with_wcc ->", out2)
else:
    print("Skipped run (RUN_DESTRUCTIVE=False). Communities already materialized.")
    # Inspect existing communities instead:
    sample = client.conn.runInstalledQuery  # already-installed query call site
    print("com_size distribution (sample of accounts):")
    df = client.conn.getVertexDataframe("Account", select="com_id,com_size", limit=5)
    print(df)

## 4 · Entity resolution

`match_parties` writes `Same_As` edges (scored party↔party matches), then
`unify_parties` collapses connected parties into `Resolved_Entity` vertices via
`Party_In_Entity`. **Mutates** entity edges/vertices — gated.

In [ ]:
for key in ("match_parties", "unify_parties"):
    log = gsql_install.install_query(client, key)
    print(f"installed {key}")
    show(log, 200)
    print("-" * 40)

In [ ]:
if RUN_DESTRUCTIVE:
    m = gsql_install.run_query(client, "match_parties")
    print("match_parties ->", str(m)[:300])
    u = gsql_install.run_query(client, "unify_parties")
    print("unify_parties ->", str(u)[:300])
else:
    n_ent = client.conn.getVertexCount("Resolved_Entity", realtime=True)
    print(f"Skipped (RUN_DESTRUCTIVE=False). Resolved_Entity count: {n_ent}")

## 5 · Feature queries

Seven account-level feature queries. All **overwrite attributes** on existing
Accounts (idempotent), so re-running is safe. They have no ordering dependency
on each other, but all should run before deriving the temporal spec.

In [ ]:
FEATURE_QUERIES = [
    "pagerank",
    "fastrp",
    "money_flow",
    "temporal_features",
    "identity_sharing",
    "triangle_clustering",
    "account_account_degree",
]

# Install all feature queries (resolves each to its real installed name).
logs = gsql_install.install_queries(client, FEATURE_QUERIES)
for key in FEATURE_QUERIES:
    print(f"installed {key:24s} -> {gsql_install.installed_query_name(key)}")

In [ ]:
# Run them. Each writes its features onto Account vertices. Safe to re-run.
# (Some are heavy full-graph scans — expect minutes each on the full graph.)
RUN_FEATURES = False  # flip to True to actually (re)compute features
if RUN_FEATURES:
    for key in FEATURE_QUERIES:
        out = gsql_install.run_query(client, key)
        print(f"{key:24s} done; sample output: {str(out)[:120]}")
else:
    print("Skipped feature runs (RUN_FEATURES=False). Features already on the graph.")
    df = client.conn.getVertexDataframe(
        "Account", select="pagerank,aa_degree,triangle_count,out_amount", limit=5
    )
    print(df)

## 6 · Derive temporal spec & reference epoch

NOT hardcoded — read from the graph. `derive_temporal_spec` scans HAS_PAID edges
for `max_bins` (and cross-checks list lengths); `edge_dim` follows from it.
`derive_reference_epoch_s` is the latest transaction time (recency anchor).

In [ ]:
from mule_pattern_learner.tigergraph.derivation import (
    derive_temporal_spec, derive_reference_epoch_s,
)

spec = derive_temporal_spec(client)
reference_epoch_s = derive_reference_epoch_s(client)
print(f"max_bins  = {spec.max_bins}")
print(f"edge_dim  = {spec.edge_dim}   (= flat_edge_dim(max_bins))")
print(f"reference_epoch_s = {reference_epoch_s:.0f}")

## 7 · Masking — SYNTHETIC ONLY

Applies the PU mask (`pu_label` / `true_label` / `bucket`) and the party-grouped,
**positive-stratified** split, then writes `is_train/is_val/is_test/pu_label` back
to Accounts and the answer-key parquet. **This step does not exist in production**
(real labels replace it). Run via the script so its CLI args apply.

> With the stratified split, val is guaranteed revealed positives. Use
> `--reveal-prevalence 0.10` for a healthier count.

In [ ]:
# Masking is synthetic scaffolding with a CLI; invoke the script rather than
# importing its main(). This MUTATES split flags + writes the parquet.
RUN_MASKING = False  # flip to True to (re)mask
if RUN_MASKING:
    import subprocess
    cmd = [sys.executable, "scripts/pipeline/after_load/masking.py",
           "--reveal-prevalence", "0.10"]
    print("running:", " ".join(cmd))
    res = subprocess.run(cmd, capture_output=True, text=True)
    print(res.stdout[-2000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-1000:])
else:
    print("Skipped (RUN_MASKING=False). Split flags + parquet already present.")

## 8 · Inspect the split seeds (the fail-fast check, interactively)

Before training, confirm each split has revealed positives — the exact guard
`train.py` enforces. With the stratified split, **val pos should be ≥ 1**.

In [ ]:
from mule_pattern_learner.training.seeds_source import fetch_split_seeds

for split in ("train", "val", "test"):
    s = fetch_split_seeds(client, split)
    print(f"{split:5s}: {len(s.account_ids):>7} accounts | "
          f"revealed positives = {s.num_positives}")
# If val positives == 0, re-mask (cell 7) with stratify + higher prevalence.

## 9 · Training smoke (a few real batches)

A miniature of `mule-train`: build loaders, run a couple of epochs to confirm the
wiring (forward → nnPU loss → backward → PAUC validation → best-checkpoint save).
For a full run, use the `mule-train --estimated-mules N` console script instead.

In [ ]:
# Derive pi the same way the entrypoint does (estimated total mules / accounts).
ESTIMATED_MULES = 216  # synthetic "we guessed right"; production = your estimate
total_accounts = client.conn.getVertexCount("Account", realtime=True)
assert isinstance(total_accounts, int)
prior = ESTIMATED_MULES / total_accounts
print(f"pi = {prior:.6f}  ({ESTIMATED_MULES} / {total_accounts})")

In [ ]:
# This block mirrors train.main() at small scale. It is intentionally a SMOKE:
# 2 epochs, so you can watch one train+val cycle. Full training => mule-train.
RUN_TRAIN_SMOKE = False  # flip on to actually train a couple epochs
if RUN_TRAIN_SMOKE:
    import math
    from collections.abc import Iterable, Iterator
    from typing import cast
    from torch_geometric.data import HeteroData
    from mule_pattern_learner.pyg.backend import TigerGraphRemoteBackend
    from mule_pattern_learner.pyg.model import MulePatternModel
    from mule_pattern_learner.pyg.neighbors import NeighborFanout
    from mule_pattern_learner.device import select_device
    from mule_pattern_learner.training.loop import TrainConfig, fit
    from mule_pattern_learner.training.seeds import SeedPool, epoch_batches

    backend = TigerGraphRemoteBackend(client)
    mapper = backend.mapper
    device = select_device()
    train_seeds = fetch_split_seeds(client, "train")
    val_seeds = fetch_split_seeds(client, "val")
    pu_label_of = dict(train_seeds.pu_label_of); pu_label_of.update(val_seeds.pu_label_of)
    pool = SeedPool(
        positives=tuple(a for a, y in train_seeds.pu_label_of.items() if y == 1),
        unlabeled=tuple(a for a, y in train_seeds.pu_label_of.items() if y == 0),
    )
    fanout = NeighborFanout()
    def to_ids(ints): return mapper.to_strings("Account", ints)
    def train_loader():
        for b in epoch_batches(pool, batch_size=512, positives_per_batch=32, seed=1337):
            yield from cast(Iterable[HeteroData], backend.make_loader(
                seed_ids=b, reference_epoch_s=reference_epoch_s, max_bins=spec.max_bins,
                fanout=fanout, batch_size=len(b), shuffle=False,
                allow_val=False, allow_test=False))
    def val_loader():
        yield from cast(Iterable[HeteroData], backend.make_loader(
            seed_ids=val_seeds.account_ids, reference_epoch_s=reference_epoch_s,
            max_bins=spec.max_bins, fanout=fanout, batch_size=512, shuffle=False,
            allow_val=True, allow_test=False))
    model = MulePatternModel(account_in_dim=31, edge_dim=spec.edge_dim)
    cfg = TrainConfig(prior=prior, max_epochs=2, patience=99, eval_k=50)
    result = fit(model=model, make_train_loader=train_loader, make_val_loader=val_loader,
                 pu_label_of=pu_label_of, mapper_to_ids=to_ids, config=cfg, device=device)
    print("best epoch:", result.best_epoch, "best val PAUC:", round(result.best_val_pauc, 4))
else:
    print("Skipped (RUN_TRAIN_SMOKE=False). Use `mule-train --estimated-mules 216` for a full run.")

## 10 · Hidden-mule evaluation — SYNTHETIC ONLY

After a real `mule-train` run produces a checkpoint, score generalization to
HIDDEN mules against the answer-key parquet. Run the script (auto-discovers the
latest checkpoint + parquet).

In [ ]:
RUN_EVAL = False  # flip on AFTER you have a checkpoint in models/
if RUN_EVAL:
    import subprocess
    res = subprocess.run([sys.executable, "scripts/experiments/evaluate_hidden.py"],
                         capture_output=True, text=True)
    print(res.stdout[-2000:])
    if res.returncode != 0:
        print("STDERR:", res.stderr[-1000:])
else:
    print("Skipped (RUN_EVAL=False). Train first, then flip on.")

---
### Refactor notes (things to look at while exploring)

- **`gsql_install`** now centralizes installs. The 7 old `_gsql` helpers in
  `scripts/experiments/` & `scripts/demos/` can be deleted and re-pointed here.
- **Registry vs installed names**: cell 1 shows the 10 mismatches. A future
  cleanup could store the installed name in `gsql_paths` directly.
- **The destructive gates** (`RUN_DESTRUCTIVE`, `RUN_FEATURES`, `RUN_MASKING`,
  `RUN_TRAIN_SMOKE`, `RUN_EVAL`) are the seams a real orchestrator would turn
  into ordered, idempotent pipeline steps (Makefile target or Python runner).